# Lab03 — Build Artifacts & Deploy to Hugging Face Spaces
Chạy từng cell theo thứ tự. Chỉ cần chạy 1 lần là build xong và deploy luôn.

## Cell 1 — Cấu hình (SỬA Ở ĐÂY TRƯỚC KHI CHẠY)

In [ ]:
# ============================================================
# SỬA 2 GIÁ TRỊ NÀY TRƯỚC KHI CHẠY
# ============================================================
HF_TOKEN   = "hf_xxxxxxxxxxxxxxxxxxxx"   # token HF có quyền write
HF_REPO_ID = "dangvanky/lab03-gnn-graphrag-dvk"  # username/space-name

# Tham số build (có thể giữ nguyên)
MAX_RECORDS = 1500
GNN_EPOCHS  = 60
EVAL_LIMIT  = 200

GITHUB_REPO = "https://github.com/dvk26/Lab03.git"
REPO_DIR    = "/content/lab03"
# ============================================================

## Cell 2 — Clone repo từ GitHub

In [ ]:
import os

os.system(f"rm -rf {REPO_DIR}")
os.system(f"git clone {GITHUB_REPO} {REPO_DIR}")
os.chdir(REPO_DIR)
print("Commit:", os.popen("git rev-parse --short HEAD").read().strip())

## Cell 3 — Cài thư viện cần thiết để build (không cần llama-cpp)

In [ ]:
%%capture
# Chỉ cài những gì cần cho build_all.py — bỏ qua llama-cpp và gradio
!pip install -q \
    datasets \
    huggingface-hub \
    llama-index-core \
    networkx \
    numpy \
    pandas \
    scikit-learn \
    "sentence-transformers>=3.0.0" \
    torch \
    torch-geometric \
    tqdm

!pip install -q -e .   # cài lab03 package

print("Cài xong!")

## Cell 4 — Build artifacts + Train GNN

In [ ]:
import os

os.environ["MAX_RECORDS"] = str(MAX_RECORDS)
os.environ["GNN_EPOCHS"]  = str(GNN_EPOCHS)
os.environ["EVAL_LIMIT"]  = str(EVAL_LIMIT)

# Build graph + embeddings + train GNN (tất cả trong 1 lệnh)
!python scripts/build_all.py

## Cell 5 — Kiểm tra artifacts đã có đủ chưa

In [ ]:
from pathlib import Path
import json

required = [
    "artifacts/graph_nodes.json",
    "artifacts/graph_edges.json",
    "artifacts/manifest.json",
    "artifacts/raw_embeddings.npy",
    "artifacts/edge_index.npy",
    "artifacts/structural_embeddings.npy",
]

print("=== Kiểm tra artifacts ===")
all_ok = True
for f in required:
    exists = Path(f).exists()
    size   = Path(f).stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {f}  ({size/1024:.1f} KB)")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("Tất cả artifacts đã có — sẵn sàng deploy!")
else:
    print("THIẾU FILE — build_all.py chưa chạy xong hoặc bị lỗi.")

# In metrics nếu có
metrics_path = Path("evaluation/retrieval_metrics.json")
if metrics_path.exists():
    print("\n=== Retrieval Metrics ===")
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))

## Cell 6 — Deploy lên Hugging Face Space

In [ ]:
from huggingface_hub import HfApi, login

login(token=HF_TOKEN, add_to_git_credential=False)

api = HfApi()

print(f"Đang upload lên {HF_REPO_ID} ...")
api.upload_folder(
    folder_path=REPO_DIR,
    repo_id=HF_REPO_ID,
    repo_type="space",
    ignore_patterns=[
        ".git/*",
        ".venv/*",
        "**/__pycache__/*",
        "**/*.pyc",
        "**/*.pyo",
        ".env",
        "models/*",          # model GGUF sẽ tự download lúc chạy
        ".pytest_cache/*",
    ],
)

print(f"\nDone! Space đang rebuild tại:")
print(f"https://huggingface.co/spaces/{HF_REPO_ID}")